In [4]:
# Step 1 — Load CSV / Documents
import pandas as pd

df = pd.read_csv("Training_Dataset.csv")
texts = df["instruction"].tolist()

ParserError: Error tokenizing data. C error: EOF inside string starting at row 24604

In [4]:
#Step 2 — Chunking
def simple_chunk(instruction, chunk_size=300):
    return [instruction[i:i+chunk_size] for i in range(0, len(instruction), chunk_size)]

chunks = []
for t in texts:
    chunks.extend(simple_chunk(t))

In [5]:
#Step 3 — Embeddings
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/840 [00:00<?, ?it/s]

In [13]:
# Step 4 — Vector DB (FAISS)
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [14]:
# Step 5 — Retrieval function
def retrieve(query, k=3):
    q_emb = model.encode([query])
    distances, indices = index.search(np.array(q_emb), k)
    return [chunks[i] for i in indices[0]]

In [19]:
# Step 6 — Load Qwen3

# Example using HuggingFace:

from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [1]:
#Step 7 — RAG Prompt
def generate_answer(question):
    docs = retrieve(question)

    context = "\n\n".join(docs)

    prompt = f"""
You are a customer support assistant.

Use the context below to answer the question.

Context:
{context}

Question:
{question}

Answer clearly and concisely:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    output = llm.generate(**inputs, max_new_tokens=300)

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [2]:
print(generate_answer("How can I track my order?"))

NameError: name 'retrieve' is not defined

In [ ]:
# QWEN without RAG
def test_qwen(question):
    prompt = f"""
You are a helpful customer support assistant.

Question:
{question}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test without RAG
print(test_qwen("How can I track my order?"))